In [ ]:
!pip install requests

In [ ]:
!pip install pypdf -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.1/385.1 kB 6.5 MB/s eta 0:00:00


In [ ]:
from google.colab import files
uploaded = files.upload()
INPUT_CSV= list(uploaded.keys())[0] #shows the actyal name it is saved as in colab, saves the csv as a list
print("Using file:", INPUT_CSV) #get cst_corpus_173.csv (my file, containing the full 173 texts with domain)


KeyboardInterrupt: 

In [ ]:
import csv #for reading csv files
import io #treat downloaded PDF bytes as an in-memory "file" without saving to disk first
import re #regular expression, used for comapring titles
import sys #used to fix the CSV field size limit bug
import time #for tim.sleep()to pause between requests so we don't hammer between APIs
import difflib #to compare the similarity between twos trings, to compare arXiv titles
from dataclasses import dataclass #to make simple classes that just hold data
from typing import Optional
import requests #to call HTTP calls to dowload pdfs
import xml.etree.ElementTree as ET #parses XML, since arXiv's API returns XML, not JSON

try:
    from pypdf import PdfReader
except ImportError:
    raise SystemExit("Missing dependency. Run: pip install pypdf --break-system-packages")

csv.field_size_limit(min(sys.maxsize, 2**31 - 1)) #maxsize= largest integer Python can comfortably use in the system, to prevent cvs reader from faling

POLITE_EMAIL = "lamihaazad@gmail.com" #openalex askas for a contact email fro faster
REQUEST_TIMEOUT = 20
MAX_PDF_BYTES = 50 * 1024 * 1024 #cap,so it doesnt use too much memory
BROWSER_HEADERS = { #in case, bcs some sites block requests that dont look like real browser
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept": "application/pdf,text/html,*/*",
}

In [ ]:
class Candidate:
   #location for papers full text, could resolve to PDF
    def __init__(self, source_label, url=None, text=None): #URL that still neeed dowloading
        self.source_label = source_label
        self.url = url
        self.text = text


class ExtractionResult: #extracting from url, pdf, html or dowload failed
    def __init__(self, status, text="", reason=""):
        self.status = status
        self.text = text
        self.reason = reason


class PaperRecord:
    #puts one row of the corpus CSV togetherwith full text processign state
    def __init__(self, row):
        self.row = row
        self.fulltext= row.get("fulltext", "")
        self.fulltext_status= row.get("fulltext_status", "NOT_ATTEMPTED")
        self.fulltext_source= row.get("fulltext_source", "")
        self.source_url_used = row.get("source_url_used", "")
        self.failure_reason= row.get("failure_reason", "")

    @property
    def doi(self):
        return self.row.get("doi", "").strip()
    @property
    def title(self):
        return self.row.get("title", "")

    def already_has_fulltext(self):
        return self.fulltext_status == "PDF_OK" and self.fulltext

    def is_closed_access(self):
        return self.row.get("is_oa", "").strip().upper() != "TRUE"

    def apply_result(self, status, text, reason, source_label, url_used):
        self.fulltext_status = status
        self.fulltext = text
        self.failure_reason = reason
        self.fulltext_source = source_label
        self.source_url_used = url_used

    def to_output_row(self):
        out = dict(self.row)
        out["fulltext_status"]= self.fulltext_status
        out["fulltext_source"]= self.fulltext_source
        out["source_url_used"]= self.source_url_used
        out["failure_reason"]= self.failure_reason
        out["fulltext"] = self.fulltext
        return out

In [ ]:
class PdfExtractor:
    def __init__(self, timeout=REQUEST_TIMEOUT, max_bytes=MAX_PDF_BYTES):
        self.timeout = timeout
        self.max_bytes = max_bytes

    def extract(self, url: str) -> ExtractionResult:
        try:
            r = requests.get(url, timeout=self.timeout, headers=BROWSER_HEADERS,
                              stream=True, allow_redirects=True)
        except requests.exceptions.Timeout:
            return ExtractionResult("DOWNLOAD_FAILED", reason="timeout")
        except requests.RequestException as e:
            return ExtractionResult("DOWNLOAD_FAILED", reason=type(e).__name__)

        if r.status_code != 200:
            return ExtractionResult("DOWNLOAD_FAILED", reason=f"HTTP {r.status_code}")

        content = self._read_capped(r)
        if content is None:
            return ExtractionResult("DOWNLOAD_FAILED", reason="exceeded size cap")

        content_type = r.headers.get("Content-Type", "").lower() #some sites dont set content type right so also check the actual bytes
        if not ("application/pdf" in content_type or content[:5] == b"%PDF-"):
            return ExtractionResult("HTML_ONLY", reason=f"content-type: {content_type or 'unknown'}")

        return self._parse_pdf(content)

    def _read_capped(self, response): #to not download some 500mb file accidentaly, stop before fully downloaded
        chunks, total = [], 0
        for chunk in response.iter_content(chunk_size=8192):
            total += len(chunk)
            if total > self.max_bytes:
                return None
            chunks.append(chunk)
        return b"".join(chunks)

    def _parse_pdf(self, content: bytes) -> ExtractionResult:
        try:
            reader = PdfReader(io.BytesIO(content))
        except Exception as e:
            return ExtractionResult("PARSE_FAILED", reason=f"pypdf error: {type(e).__name__}")

        pages_text = []
        for page in reader.pages:
            try:
                pages_text.append(page.extract_text() or "")
            except Exception: #so one bad page doesnt stop th eprocess
                continue
        text = "\n".join(pages_text).strip()
        if text:
            return ExtractionResult("PDF_OK", text=text)
        return ExtractionResult("PARSE_FAILED", reason="no extractable text (likely scanned image)")




In [ ]:
class BaseSource:
    label = "base"

    def get_candidates(self, record: PaperRecord) -> list:
        raise NotImplementedError

    def throttle(self):

        time.sleep(0.15) #default delay so dont hammer the apis


class StoredUrlSource(BaseSource): #this one doesnt need to call anything, url already in the csv

    label = "stored_oa_url"

    def get_candidates(self, record: PaperRecord) -> list:
        url = record.row.get("oa_url", "")
        return [Candidate(self.label, url=url)] if url else []

    def throttle(self):
        pass  # when nothing fetched, nothing to throttle


class OpenAlexSource(BaseSource):
    label = "openalex"
    URL = "https://api.openalex.org/works/https://doi.org/{doi}"

    def get_candidates(self, record: PaperRecord) -> list:
        url = self.URL.format(doi=record.doi) + f"?mailto={POLITE_EMAIL}"
        try:
            r = requests.get(url, timeout=REQUEST_TIMEOUT)
            if r.status_code != 200:
                return []
            data = r.json()
        except requests.RequestException:
            return []
#collect every pdf orl oPENALEX offers like best location, prim location, any other, so later it has another option to fall back to if it doesn work
        urls = []
        best = data.get("best_oa_location") or {}
        if best.get("pdf_url"):
            urls.append(best["pdf_url"])
        primary = data.get("primary_location") or {}
        if primary.get("pdf_url"):
            urls.append(primary["pdf_url"])
        for loc in data.get("locations", []) or []:
            if loc.get("pdf_url"):
                urls.append(loc["pdf_url"])

        seen, deduped = set(), []
        for u in urls:
            if u not in seen:
                seen.add(u)
                deduped.append(u)
        return [Candidate(self.label, url=u) for u in deduped]


class SemanticScholarSource(BaseSource):
    label = "semanticscholar"
    URL = "https://api.semanticscholar.org/graph/v1/paper/DOI:{doi}?fields=openAccessPdf"

    def get_candidates(self, record: PaperRecord) -> list:
        url = self.URL.format(doi=record.doi)
        try:
            r = requests.get(url, timeout=REQUEST_TIMEOUT)
            if r.status_code != 200:
                return []
            data = r.json()
        except requests.RequestException:
            return []

        pdf_info = data.get("openAccessPdf") or {}
        pdf_url = pdf_info.get("url")
        return [Candidate(self.label, url=pdf_url)] if pdf_url else []


class CoreSource(BaseSource):
    """CORE hosts its OWN copies of many papers, so it bypasses publisher blocking."""
    label = "core"
    SEARCH_URL = "https://api.core.ac.uk/v3/search/works"

    def __init__(self, api_key: str = ""):
        self.api_key = api_key

    def get_candidates(self, record: PaperRecord) -> list:
        if not self.api_key:
            return []
        try:
            r = requests.get(
                self.SEARCH_URL,
                params={"q": f'doi:"{record.doi}"'},
                headers={"Authorization": f"Bearer {self.api_key}"},
                timeout=REQUEST_TIMEOUT,
            )
            if r.status_code == 429:
                time.sleep(10)
                return []
            if r.status_code != 200:
                return []
            data = r.json()
        except requests.RequestException:
            return []

        results = data.get("results", [])
        if not results:
            return []
        work = results[0]
        candidates = []
        fulltext = (work.get("fullText") or "").strip()
        if fulltext:
            candidates.append(Candidate(self.label, text=fulltext))
        download_url = work.get("downloadUrl")
        if download_url:
            candidates.append(Candidate(self.label, url=download_url))
        return candidates

    def throttle(self):
        time.sleep(3)  # CORE's free tier is rate-limited


class ArxivSource(BaseSource):
    #Matched by title similarity, since arXiv search doesn't reliably index external DOIs
    label = "arxiv"
    API_URL = "http://export.arxiv.org/api/query"
    MIN_TITLE_SIMILARITY = 0.90

    def get_candidates(self, record: PaperRecord) -> list:
        title = record.title
        if not title:
            return []
        try:
            r = requests.get(
                self.API_URL,
                params={"search_query": f'ti:"{title}"', "max_results": 1},
                timeout=REQUEST_TIMEOUT,
            )
            if r.status_code != 200:
                return []
            ns = {"atom": "http://www.w3.org/2005/Atom"}
            root = ET.fromstring(r.text)
            entry = root.find("atom:entry", ns)
            if entry is None:
                return []
            found_title = entry.find("atom:title", ns).text or ""
        except (requests.RequestException, ET.ParseError):
            return []

#this is to guars agaisnt a flse match if arxiv title search is loos and verify the returned title is what i was lookign for
        if self._similarity(title, found_title) < self.MIN_TITLE_SIMILARITY:
            return []

        entry_id = entry.find("atom:id", ns).text or ""
        arxiv_id = entry_id.rstrip("/").split("/")[-1]
        return [Candidate(self.label, url=f"https://arxiv.org/pdf/{arxiv_id}")]

    @staticmethod
    def _normalize(t):
        return re.sub(r"[^a-z0-9 ]", "", (t or "").lower()).strip()

    def _similarity(self, a, b):
        return difflib.SequenceMatcher(None, self._normalize(a), self._normalize(b)).ratio()

    def throttle(self):
        time.sleep(1)

In [ ]:
class FullTextPipeline:
    def __init__(self, sources: list, extractor: PdfExtractor = None,
                 skip_closed_access: bool = True, resume: bool = True):
        self.sources = sources
        self.extractor = extractor or PdfExtractor()
        self.skip_closed_access = skip_closed_access
        self.resume = resume  # if True, don't re-attempt rows that already have PDF_OK

    def run(self, input_csv: str, output_csv: str):
        records = self._load_records(input_csv)
        to_process = [r for r in records if self._needs_processing(r)]
        print(f"Total rows: {len(records)} | To process: {len(to_process)}")

        rescued = 0
        for i, record in enumerate(to_process, 1):
            print(f"[{i}/{len(to_process)}] ref {record.row.get('ref_num')}: {record.doi}")
            if self._process_record(record):
                rescued += 1

        self._write_records(records, output_csv)

        print("\n--- Summary ---")
        print(f"Processed: {len(to_process)}")
        print(f"Full text obtained this run: {rescued}")
        print(f"Still without full text: {len(to_process) - rescued}")
        print(f"Saved to {output_csv}")

    def _needs_processing(self, record: PaperRecord) -> bool:
        if self.skip_closed_access and record.is_closed_access():
            return False
        if self.resume and record.already_has_fulltext():
            return False
        return True

    def _process_record(self, record: PaperRecord) -> bool:

      #walk sources in priority ans  order and stop the first one that yields usable text
        for source in self.sources:
            candidates = source.get_candidates(record)
            source.throttle()

            for candidate in candidates:
                if candidate.text:
                    record.apply_result("PDF_OK", candidate.text, "", source.label, "")
                    return True

                if candidate.url:
                    result = self.extractor.extract(candidate.url)
                    if result.status == "PDF_OK":
                        record.apply_result("PDF_OK", result.text, "", source.label, candidate.url)
                        return True
                    # keep the most recent failure reason if case nothing pans out- row missing tho
                    record.failure_reason = result.reason
                    if result.status == "HTML_ONLY" and not record.source_url_used:
                        record.source_url_used = candidate.url

        # if nothing worked
        if not record.fulltext_status or record.fulltext_status == "NOT_ATTEMPTED":
            record.fulltext_status = "HTML_ONLY" if record.source_url_used else "NOT_FOUND"
        return False

    def _load_records(self, input_csv: str) -> list:
        with open(input_csv, newline="", encoding="utf-8") as f:
            rows = list(csv.DictReader(f))
        return [PaperRecord(row) for row in rows]

    def _write_records(self, records: list, output_csv: str):
        out_rows = [r.to_output_row() for r in records]
        out_rows.sort(key=lambda r: int(r["ref_num"]))

        base_fields = ["ref_num", "doi","title","abstract","year", "source",
                       "is_oa", "oa_status", "oa_url"]

                       #still keeping these unexpected columns in case
        extra_fields = ["fulltext_status", "fulltext_source","source_url_used",
                        "failure_reason","fulltext"]
        seen_keys = set()
        for r in out_rows:
            seen_keys.update(r.keys())
        fieldnames= [f for f in base_fields + extra_fields if f in seen_keys]
        fieldnames += [k for k in seen_keys if k not in fieldnames]

        for r in out_rows:
            for f in fieldnames:
                r.setdefault(f, "")

        with open(output_csv, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(out_rows)

In [ ]:
if __name__ == "__main__":
    CORE_API_KEY = " "  #paste the free CORE API key here (mail)

    # if u dont have a core api it will just skip this stage
    try:
        INPUT_CSV
    except NameError:
        INPUT_CSV = "cst_corpus_with_abstracts.csv"

    OUTPUT_CSV = "cst_corpus_with_fulltext_oop.csv"

    pipeline = FullTextPipeline(sources=[
        StoredUrlSource(),
        OpenAlexSource(),
        SemanticScholarSource(),
        CoreSource(api_key=CORE_API_KEY),
        ArxivSource(),
    ])
    pipeline.run(input_csv=INPUT_CSV, output_csv=OUTPUT_CSV)


In [ ]:
import csv
import sys

csv.field_size_limit(min(sys.maxsize, 2**31 - 1))

RESULTS_CSV = "cst_corpus_with_fulltext_oop.csv"

with open(RESULTS_CSV, newline="", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

print("Columns:", list(rows[0].keys()))

unresolved = [r for r in rows if r["fulltext_status"] in ("NOT_FOUND", "HTML_ONLY")]
print(f"Found {len(unresolved)} unresolved rows")

dois = [r["doi"].strip() for r in unresolved]
print("\n".join(dois))

with open("unresolved_dois.txt", "w") as f:
    f.write("\n".join(dois))

In [ ]:
from google.colab import files
files.download("unresolved_dois.txt")

ZOTERO CORPUS RETRIEVAL

In [ ]:
from google.colab import files

print("Upload cst_corpus_with_fulltext_oop.csv:")
uploaded = files.upload()
INPUT_CSV = list(uploaded.keys())[0]

print("Upload zotero_Exported_Items.zip:")
uploaded2 = files.upload()
ZOTERO_ZIP = list(uploaded2.keys())[0]

Zotero PDF merge: 2 SECT


In [ ]:
# --- Zotero PDF merge: shared implementation ---
#
# This defines the RDF-parsing / PDF-extraction / CSV-merging logic used by
# BOTH Zotero passes in this notebook (this cell, and the closed-access pass
# further down).


import csv
import re
import shutil
import sys
import zipfile
import unicodedata
import xml.etree.ElementTree as ET
from pathlib import Path

from pypdf import PdfReader

csv.field_size_limit(min(sys.maxsize, 2**31 - 1))

NS = {
    "rdf": "http://www.w3.org/1999/02/22-rdf-syntax-ns#",
    "z": "http://www.zotero.org/namespaces/export#",
    "dc": "http://purl.org/dc/elements/1.1/",
    "dcterms": "http://purl.org/dc/terms/",
    "bib": "http://purl.org/net/biblio#",
    "link": "http://purl.org/rss/1.0/modules/link/",
}
DOI_RE = re.compile(r"10\.\d{4,9}/[^\s\"'<>]+", re.IGNORECASE)


class ZoteroRdfParser:
   #Parses  Zotero RDF export into doi: local pdf path mapping

    def __init__(self, zip_path: str, extract_to: str = "zotero_export"):
        self.zip_path= zip_path
        self.extract_to= Path(extract_to)
        self.rdf_dir= None
        self.rdf_file= None

    def unzip(self):
        if self.extract_to.exists():
            shutil.rmtree(self.extract_to)

        with zipfile.ZipFile(self.zip_path, "r") as z:
            for info in z.infolist():
                # Zip entries created on macOS often store filenames as raw
                # CP437 bytes instead of UTF-8 (the flag bit below marks
                # "real" UTF-8 names). Re-decode as CP437->UTF-8 so accented
                # filenames extract with matching names instead of mojibake;
                # fall back to the original name if that decode fails.
                if not info.flag_bits & 0x800:
                    try:
                        info.filename = info.filename.encode("cp437").decode("utf-8")
                    except (UnicodeDecodeError, UnicodeEncodeError):
                        pass
                z.extract(info, self.extract_to)

        self.rdf_dir = next(
            d for d in self.extract_to.glob("*/") if d.name != "__MACOSX"
        ).resolve()
        self.rdf_file = next(self.rdf_dir.glob("*.rdf"))
        return self

    def build_doi_to_pdf_map(self) -> dict:
        if self.rdf_file is None:
            self.unzip()

        tree= ET.parse(self.rdf_file)
        root= tree.getroot()

        attachment_paths = self._get_attachment_paths(root)

        doi_to_pdf = {}
        for elem in root:
            # Zotero represents different item types with different top-level
            # RDF tags (rdf:Description for most, bib:Article/bib:Journal for
            # others), so need check every direct child rather than one tag name.
            doi = self._extract_doi(elem)
            if not doi:
                continue
            link_elem = elem.find("link:link", NS)
            if link_elem is None:
                continue
            target = link_elem.get("{http://www.w3.org/1999/02/22-rdf-syntax-ns#}resource")
            if target in attachment_paths:
                pdf_path = self._resolve_pdf_path(attachment_paths[target])
                if pdf_path.exists():
                    doi_to_pdf[doi.lower()] = pdf_path

        return doi_to_pdf

    def _get_attachment_paths(self, root) -> dict:
        attachment_paths = {}
        for att in root.findall("z:Attachment", NS):
            about = att.get("{http://www.w3.org/1999/02/22-rdf-syntax-ns#}about")
            path_elem = att.find("z:path", NS)
            if about and path_elem is not None:
                resource = path_elem.get("{http://www.w3.org/1999/02/22-rdf-syntax-ns#}resource")
                if resource:
                    attachment_paths[about] = resource
        return attachment_paths

    def _extract_doi(self, elem):
        for identifier in elem.findall(".//dc:identifier", NS):
            text = "".join(identifier.itertext())
            m = DOI_RE.search(text)
            if m:
                return m.group(0).rstrip(".,;")
        return None

    def _resolve_pdf_path(self, relative_path: str) -> Path:
        #The RDF file's stated path may use NFC unicode (precomposed accented
        #characters), while filenames extracted from a macOS-created zip are
        #often NFD (decomposed)  same visible text, different bytes. Try the
        #direct path first, then fall back to matching by normalized form.
        direct = self.rdf_dir / relative_path
        if direct.exists():
            return direct

        target_name_nfc = unicodedata.normalize("NFC", Path(relative_path).name)
        parent_dir = self.rdf_dir / Path(relative_path).parent
        if parent_dir.is_dir():
            for candidate in parent_dir.iterdir():
                if unicodedata.normalize("NFC", candidate.name) == target_name_nfc:
                    return candidate
        return direct


class PdfTextExtractor:
    """Pulls plain text out of a local PDF file, page by page."""

    def extract(self, pdf_path: Path) -> str:
        try:
            reader = PdfReader(str(pdf_path))
        except Exception as e:
            print(f"  ! Failed to open {pdf_path.name}: {e}")
            return ""

        pages_text = []
        for page in reader.pages:
            try:
                pages_text.append(page.extract_text() or "")
            except Exception:
                # A single unparsable page shouldn't sink the whole document --
                # keep whatever text the other pages yielded.
                continue

        text = "\n".join(pages_text).strip()
        return self._sanitize(text)

    def _sanitize(self, text: str) -> str:
        # pypdf sometimes emits lone unicode surrogates from broken PDF font
        # encodings, which can't be written to a normal UTF-8 file -- drop them.
        return text.encode("utf-8", errors="ignore").decode("utf-8", errors="ignore")


class CorpusMerger:
    #Fills in missing fulltext cells in the corpus CSV from local Zotero PDFs

    def __init__(self, doi_to_pdf: dict, extractor: PdfTextExtractor = None,
                 source_label: str = "zotero", target_statuses: set = None):
        self.doi_to_pdf = doi_to_pdf
        self.extractor = extractor or PdfTextExtractor()
        self.source_label = source_label
        #  fulltext_status values are eligible for a fill attempt.


        self.target_statuses = target_statuses
        self.updated_count = 0
        self.still_missing = []

    def merge(self, input_csv: str, output_csv: str):
        with open(input_csv, newline="", encoding="utf-8") as f:
            rows = list(csv.DictReader(f))

        for row in rows:
            status = row.get("fulltext_status")
            if status== "PDF_OK":
                continue
            if self.target_statuses is not None and status not in self.target_statuses:
                continue
            self._try_fill_row(row)

        self._write(rows, output_csv)
        self._print_summary(output_csv)

    def _try_fill_row(self, row: dict):
        doi = row["doi"].strip().lower()
        pdf_path = self.doi_to_pdf.get(doi)

        if not pdf_path:
            self.still_missing.append(doi)
            return

        print(f"Extracting: {pdf_path.name}")
        text = self.extractor.extract(pdf_path)

        if text:
            row["fulltext"] = text
            row["fulltext_status"] = "PDF_OK"
            row["fulltext_source"] = self.source_label
            row["source_url_used"] = pdf_path.name
            row["failure_reason"] = ""
            self.updated_count += 1
        else:
            print(f"  ! No extractable text in {pdf_path.name}")
            self.still_missing.append(doi)

    def _write(self, rows: list, output_csv: str):
        fieldnames = list(rows[0].keys())
        with open(output_csv, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(rows)

    def _print_summary(self, output_csv: str):
        print("\n--- Summary ---")
        print(f"Rows updated via Zotero PDFs: {self.updated_count}")
        print(f"Still missing: {len(self.still_missing)}")
        for doi in self.still_missing:
            print(f"  {doi}")
        print(f"\nSaved to {output_csv}")


#  Pass 1: fill rows the automated pipeline explicitly gave up on
OUTPUT_CSV = "cst_corpus_with_fulltext_final.csv"

parser = ZoteroRdfParser(zip_path=ZOTERO_ZIP)
doi_to_pdf = parser.build_doi_to_pdf_map()
print(f"Found {len(doi_to_pdf)} DOI -> PDF mappings\n")

merger = CorpusMerger(
    doi_to_pdf=doi_to_pdf,
    source_label="zotero",
    target_statuses={"NOT_FOUND", "HTML_ONLY"},
)
merger.merge(input_csv=INPUT_CSV, output_csv=OUTPUT_CSV)

In [ ]:
from google.colab import files

print("Upload cst_corpus_with_fulltext_final.csv (your 94/172 file):")
uploaded = files.upload()
INPUT_CSV = list(uploaded.keys())[0]

print("Upload your new closed-access Zotero export zip:")
uploaded2 = files.upload()
ZOTERO_ZIP = list(uploaded2.keys())[0]

In [ ]:
#Zotero PASS 2
# fills remaining rows using a second (closed-access) Zotero export ---
#
# Repeats: Reuses ZoteroRdfParser / PdfTextExtractor / CorpusMerger defined in the
# shoudl run after Zotero pass 1

SOURCE_LABEL = "zotero_unpaywall"  # closed-access copies retrieved via Zotero
OUTPUT_CSV = "cst_corpus_with_fulltext_final.csv2"

parser = ZoteroRdfParser(zip_path=ZOTERO_ZIP)
doi_to_pdf = parser.build_doi_to_pdf_map()
print(f"Found {len(doi_to_pdf)} DOI -> PDF mappings\n")

merger = CorpusMerger(
    doi_to_pdf=doi_to_pdf,
    source_label=SOURCE_LABEL,
    target_statuses=None,  # broader than pass 1: also catches closed-access rows still at NOT_ATTEMPTED
)
merger.merge(input_csv=INPUT_CSV, output_csv=OUTPUT_CSV)